# Utilizei a mesma arquitetura da aula que é guiada pelo artigo "Attention is All You Need"
# Porém para diferentes datasets, e fiz a limpeza do dataset principal(wikipedia) em outro arquivo(clean_data_pratica) que ao final gera um arquivo txt onde esta pronto para ser usado

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [8]:
batch_size = 32 #quantidade de caracteres processados paralelamente
block_size = 364 # contexto que vai para o modelo
max_iters = 6000
eval_interval = 300
lr = 0.001# taxa de aprendizado do modelo
eval_iters = 200
dim_embedding = 256 #dimensao do embedding
headers = 16 # quantidade de headers que vao procurar afinidades
transformers = 8 #quantidade de seqeuncias empilhadas
dropout = 0.4
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [9]:
with open('/content/drive/MyDrive/matheus_lam/data (1).txt', "r", encoding="utf-8") as f:
    txt = f.read(25000000)

char = set(txt)

In [10]:
chs = sorted(list(char))
#cria-se uma lista ordenada contendo os carcateres do texto
vocab = len(chs)

In [11]:
list_idxs = {ch:i for i,ch in enumerate(chs)}
# pega cada caractere e trnsforma em indice em uma lista de inteiros
list_chs = {i:ch for i,ch in enumerate(chs)}
# dado um lista de interios transformada cada um em uma lista de caracteres

In [12]:
encode = lambda s: [list_idxs[c] for c in s]
# para uma string retorna seus indices(numeros) correspondentes a lista de indices
decode = lambda l: ''.join([list_chs[i] for i in l])
# decode transforma uma lista de inteiros em seus determinados caracteres

In [13]:
df = torch.tensor(encode(txt), dtype=torch.long)
# transformando o dataset inteiro em um tensor

In [14]:
df.shape

torch.Size([25000000])

In [15]:
data = int(0.9*len(df))
# pega 90% dos caracteres do texto
train = df[:data]
# separa os dados de treino e de validação
val = df[data:]


In [16]:
def get_batch(split):
    data = train if split == 'train' else val
    idx = torch.randint(len(data) - block_size, (batch_size,))
    # gera uma tupla(batch aeatorio) com base nos dados
    x = torch.stack([data[i:i+block_size]for i in idx])
    # para pega do indice 0 ate 128+1 assim tendo varios caracteres
    # cada iteração do loop sai o primeiro char e é adicionado um novo ao final
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])
    # mesma coisa para x porem pega os caracteres futuros ao x
    # o stack coloca todas os tensors e concatena todos em uma matriz, nisso para cada linha em x tem um y onde contem os caracteres futuros
    x, y = x.to(device),y.to(device)
    return x,y

In [17]:
@torch.no_grad()
def loss_estimated():
    out = {}
    model.eval()
    # coloca o modelo em modo de avaliação
    for split in ['train','val']:
        loss = torch.zeros(eval_iters)
        # cria um tensor de zeros com o tamanho de iterações na avaliação
        for k in range(eval_iters):
            x,y = get_batch(split)
            # pega um batch do split treino ou val
            logits, losses = model(x,y)
            # recebe a pontuação do modelo e seu loss
            loss[k] = losses.item()
            # adiciona o loss para a lista
        out[split] = loss.mean()
        # para cada separação dos dados cooloca a media dos loss ao longo do treino
    model.train()
    return out


In [18]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        # criando tres vetoes para cada token(key,query e value)
        self.key = nn.Linear(dim_embedding, head_size, bias=False)
        # vetor que da ao modelo o que o token atual tem para oferecer ao contexto de um token que passou, estando na mesma sequencia
        self.query = nn.Linear(dim_embedding, head_size, bias=False)
        # o que esse token procura, compara com cada key de tokens anteriores, assim criando afinidade entre os tokens
        self.value = nn.Linear(dim_embedding, head_size, bias=False)
        # o valor retornado caso haja uma afinidade forte entre as duas
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))
        # matriz com a observação de um token cortando o seus proximos tokens a partir da diagonal principal da matriz
        self.dropout = nn.Dropout(dropout)
        # zera alguns pesos de attention

    def forward(self,x):
        B,T,C = x.shape
        # B(quantidade de sequencias processadas paralelamente)
        # T(tamanho do contexto)
        # C(canais)
        k = self.key(x)
        #  matriz para representar o que cada token tem a oferecer em uma frase
        q = self.query(x)
        # uma matriz que representa "o que cada token precisa"
        # key e query servem justamente para o calculo de afinidade entre os tokens
        # por exemplo, "O cachorro branco esta correndo na avenida" a query do token 'esta' talvez precise de algum sujeito para ser completa, o token 'cachorro' tem uma key que para a query de 'esta' pode ter relação forte
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        # calculo de afinidade. Multiplica as matrizes Query e Key, assim obtem uma matriz resultante com um 'score' de relevancia entre os tokens
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        # aplica a mascara casual para que tokens futuros nao possam ser vistos
        wei = F.softmax(wei, dim=-1)
        # aplicar o softmax converte os socres de afinidade em probabilidades, os valores negativos infinitos viram 0
        wei = self.dropout(wei)
        # zera pesos
        v = self.value(x)
        # info real do token
        out = wei @ v# multiplica a matriz de afinidades com a matriz de sseus conteudos reais(media)
        return out


In [19]:
class MultiHeadAttenntion(nn.Module):
    def __init__(self, num_heads,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # cria varios headers para eu seja aprendido diferentes tipos de afinidade
        self.proj = nn.Linear(head_size * num_heads, dim_embedding)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # para cada resultado de head, pega o resultado de cada um e concatena todas
        out = self.dropout(self.proj(out))
        return out

In [20]:
class FeedFoward(nn.Module):
    def __init__(self,dim_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_emb, 4*dim_emb),
            # expande o tamanho do vetor de cada token em 4 vezes
            nn.ReLU(), # pega os valores negarivos e troca por 0
            nn.Linear(4 * dim_emb, dim_emb),
            # transforma de volta no seu tamanho original
            nn.Dropout(dropout)
        )
    def forward(self,x):
        return self.net(x)

In [21]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        # o resultado da divisão resultará no tamanho do vetor que sera processado por cada header
        self.sa = MultiHeadAttenntion(n_head, head_size)
        # instanciando a classe MultiHead
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        # normaliza os dados
        x = x + self.ffwd(self.ln2(x))
        # passa os dados para a camada de atenttion
        return x

In [22]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb_table = nn.Embedding(vocab, dim_embedding)
        # cria um vetor de dim_embedding dimensoes para cada indice de caractere
        self.position_emb_table = nn.Embedding(block_size,dim_embedding)
        # mesma coisa mas agora com a posição de cada caracter
        self.blocks = nn.Sequential(*[Block(dim_embedding, n_head = headers) for _ in range(transformers)])
        # empilha blocos de transformers em sequencia
        self.ln_f = nn.LayerNorm(dim_embedding)
        # normalização de todos os blocos
        self.lm_head = nn.Linear(dim_embedding,vocab)
        # pega o vetor de n dimensoes de cada posição e transforma em tamanho de vocabulario
        self.apply(self.init_weights)
        # aplica pesos a todas as camadas do modelo

    def init_weights(self,module):
        # atribuimos pesos ao modelo, evita com que o default desistabilize rapido
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
                # zera o bias
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0,std=0.02)

    def forward(self, idx,targets = None):
        B, T = idx.shape
        # pega o tamanho de batch e o tamanho do contexto
        token_emb = self.token_emb_table(idx)
        # pega o vetor que corresponde aos indices na matriz de vetores feito pelo o embedding
        pos_embedding = self.position_emb_table(torch.arange(T, device = device))
        # pega o vetor de embeddinf de cada posição
        x = token_emb + pos_embedding

        x = self.blocks(x)# passa x(vetores do embedding selecionados) por todas as camadas transformers
        x = self.ln_f(x)
        #normaliza
        logits = self.lm_head(x)
        # valor bruto do que o modelo atribui um socre para a sequencia
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            # dimensoes de logits
            # Quantidae de sequencias
            # TAmanho do contexto
            # Canais
            logits = logits.view(B*T,C)
            # achata para um objeto 2D necessario para que utilizemos o cross_entropy
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)
            # pega o loss aplicando a função cross_entropy
        return logits, loss
    def generate(self,idx,max_tokens):
        #recebendo um sequencia de indices
        for _ in range(max_tokens):
            # para cada iteração no loop ele olha o contexto que ja passou e gera um caractere concatenando-o ao final
            idx_new = idx[:, -block_size:]
            # tira o contexto atual para ultimos block_size tokens
            # sem isso asequencia gerada seria maior que a de pos_embedding
            logits, loss = self(idx_new)
            logits = logits[:, -1,:]
            # pegando somente a previsão bruta da ultima posição, que nesse caso seria a unica que importa
            probs = F.softmax(logits,dim=-1)
            # ao aplicar o softamx, transformamos o valo bruto em probabilidade (0.0 até 1.0)
            idx_prx = torch.multinomial(probs,num_samples=1)
            # pega aleatoriamente um caractere, dado a distribuição de probabilidades ja feita
            #  é utilizado para o modelo nem sempre pegar o ultimo caractere mais provavel
            # comentado no video que ajuda o modelo a ter mais variação em sua previsão
            idx = torch.cat((idx,idx_prx), dim=1)
            # coloca o caractere sorteado ao final
        return idx


In [23]:
model = GPT()
m = model.to(device)
# coloca o modelo para GPU

In [24]:
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')
# quantidade de parametros do modelo

6.929405 M parameters


In [25]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
# incializando o otimizador de pesos

In [26]:
lr_reducer = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
# o lr reducer reduz o learning rate caso veja que o val_loss nao reduza em 3 avaliações

In [27]:
for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        # esse if mostra somente os numeros que ao dividir ppor eval_interval seu resto seja 0, ou que seja a ultima iteração
        losses = loss_estimated()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        lr_reducer.step(losses['val'])


    xb, yb = get_batch('train')
    # pegando batch de treino e target

    logits, loss = model(xb, yb)
    # prevendo o logits e  pegando o loss
    optimizer.zero_grad(set_to_none=True)
    # zerando o gradiente
    loss.backward()
    # backpropagation
    optimizer.step()
    # atualiza os pesos

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_tokens=500)[0].tolist()))

step 0: train loss 6.9288, val loss 6.9285
step 300: train loss 2.4975, val loss 2.5078
step 600: train loss 2.4601, val loss 2.4726
step 900: train loss 2.3650, val loss 2.3807
step 1200: train loss 2.0435, val loss 2.0802
step 1500: train loss 1.8509, val loss 1.8939
step 1800: train loss 1.6992, val loss 1.7693
step 2100: train loss 1.5997, val loss 1.6788
step 2400: train loss 1.5358, val loss 1.6212
step 2700: train loss 1.4707, val loss 1.5685
step 3000: train loss 1.4386, val loss 1.5297
step 3300: train loss 1.4125, val loss 1.5206
step 3600: train loss 1.3871, val loss 1.4851
step 3900: train loss 1.3598, val loss 1.4607
step 4200: train loss 1.3456, val loss 1.4485
step 4500: train loss 1.3259, val loss 1.4255
step 4800: train loss 1.3077, val loss 1.4069
step 5100: train loss 1.3083, val loss 1.4015
step 5400: train loss 1.2886, val loss 1.3848
step 5700: train loss 1.2813, val loss 1.3816
step 5999: train loss 1.2689, val loss 1.3623

Teorial do discurso, conceu no Estado c

In [28]:
torch.save(model.state_dict(), 'pretrain.pt')
# salvando o modelo
import json
with open('vocab.json', 'w', encoding='utf-8') as f:
    json.dump({'stoi': list_idxs, 'itos': list_chs}, f, ensure_ascii=False)
    #salvando os vocabs

In [29]:
print(len(df), len(train), len(val), block_size)

25000000 22500000 2500000 364


In [30]:
len(txt)

25000000

In [31]:
with open('vocab.json', 'r', encoding='utf-8') as f:
    vocab_data = json.load(f)
# carregando o vocabuario salvo

In [32]:
list_idxs = {ch: int(i) for ch, i in vocab_data['stoi'].items()}
# pega cada caractere em  uma lista de inteiros
list_chs = {int(i): ch for i, ch in vocab_data['itos'].items()}
# dado uma lista de inteiros, retorra seu respectivo caracttere


In [33]:
vocab = len(list_idxs)
# tamanho do vocabulario

In [34]:
encode = lambda s: [list_idxs[c] for c in s]
# dado uma string retorna seus indices
decode = lambda l: ''.join([list_chs[i] for i in l])
# para uma lista de inteiros retorna seus respectivos caracteres

In [39]:
with open('/content/drive/MyDrive/matheus_lam/machado.txt', 'r', encoding='utf-8') as f:
    txt_machado = f.read()
# le o corpus do machado
print(len(txt_machado))

14562006


In [ ]:
txt_machado = txt_machado.replace('Û', '')
# a base que peguei do machado de assis, tem algumas divergencias em relação a caracteres

In [ ]:
chars_novos = set(txt_machado) - set(list_idxs.keys())
print(chars_novos)
#verificação de vocab
# verifica se há algum caractere diferente do que o modelo viu no treino, coisa que daria erro

set()


In [44]:
df_machado = torch.tensor(encode(txt_machado), dtype=torch.long)
data_mach = int(0.9*len(df_machado))
train = df_machado[:data_mach]
val = df_machado[data_mach:]
# separa os dados em treino e validação

In [45]:
# recria o modelo com os mesmos hiperparametros de arquitetura do pretraining
model = GPT()
model.load_state_dict(torch.load('pretrain.pt', map_location=device))
m = model.to(device)

In [46]:
lr =  0.00003
max_iters = 500
eval_interval = 50

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
#cria o otimizador com os params do modelo de pre learning

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = loss_estimated()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

torch.save(model.state_dict(), 'finetune.pt')

step 0: train loss 1.7074, val loss 1.7114
step 50: train loss 1.6029, val loss 1.6328
step 100: train loss 1.5696, val loss 1.5977
step 150: train loss 1.5480, val loss 1.5797
step 200: train loss 1.5352, val loss 1.5775
step 250: train loss 1.5222, val loss 1.5613
step 300: train loss 1.5068, val loss 1.5476
step 350: train loss 1.5034, val loss 1.5450
step 400: train loss 1.4910, val loss 1.5415
step 450: train loss 1.4857, val loss 1.5288
step 499: train loss 1.4808, val loss 1.5237


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_tokens=500)[0].tolist()))
# geração de um texto com o fine tunning


Viilas padrômérico. Eu a natureza da caixã requelerará também implicada de um esquelador, amandado, L. — Este senegro, depois apenas palavras tratalmente, que me antigamente, alavra, a tarde da vipulaçurrapaa moçou o peixete, no estudo ao comércio de andar à real, em ali; a resultação, como milham ele. Necidade amosteria de Io Grande, a paixão, e nas mandos saíras, cavalandou-lhe ou de comem-lhe. Se dela. Que significará uma coimbraça, surria os seus mostres. Maços. Se tobramos naturais Soares. 
